In [3]:
pip install pyzbar opencv-python pillow ipywidgets matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 MB 42.9 MB/s  0:00:01 eta 0:00:01

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [7]:
from pyzbar.pyzbar import decode
import cv2
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
import ipywidgets as widgets


# =========================
# 1. 업로드 위젯 / 실행 버튼
# =========================

uploader = widgets.FileUpload(
    accept=".png,.jpg,.jpeg",
    multiple=False
)

run_button = widgets.Button(
    description="바코드 읽기",
    button_style="success"
)

output = widgets.Output()

display(uploader)
display(run_button)
display(output)


# =========================
# 2. 업로드 이미지 읽기
# =========================

def get_uploaded_file_content(uploader):
    """
    ipywidgets 버전에 따라 uploader.value 구조가 다를 수 있어
    dict / tuple 형태를 모두 처리합니다.
    """
    if not uploader.value:
        raise ValueError("이미지를 먼저 업로드하세요.")

    # ipywidgets 7.x: dict 형태
    if isinstance(uploader.value, dict):
        uploaded_file = list(uploader.value.values())[0]
        return uploaded_file["content"]

    # ipywidgets 8.x: tuple 형태
    if isinstance(uploader.value, tuple):
        uploaded_file = uploader.value[0]
        return uploaded_file["content"]

    raise TypeError("지원하지 않는 FileUpload 데이터 형식입니다.")


def load_uploaded_image(uploader):
    image_bytes = get_uploaded_file_content(uploader)

    image_array = np.frombuffer(image_bytes, np.uint8)
    image = cv2.imdecode(image_array, cv2.IMREAD_COLOR)

    if image is None:
        raise ValueError("이미지를 읽지 못했습니다.")

    return image


# =========================
# 3. 이미지 회전
# =========================

def rotate_image(image, angle):
    if angle == 0:
        return image
    elif angle == 90:
        return cv2.rotate(image, cv2.ROTATE_90_CLOCKWISE)
    elif angle == 180:
        return cv2.rotate(image, cv2.ROTATE_180)
    elif angle == 270:
        return cv2.rotate(image, cv2.ROTATE_90_COUNTERCLOCKWISE)
    else:
        raise ValueError("angle은 0, 90, 180, 270 중 하나여야 합니다.")


# =========================
# 4. 바코드 읽기
# =========================

def read_barcode(image):
    decoded_objects = decode(image)

    results = []

    for obj in decoded_objects:
        barcode_data = obj.data.decode("utf-8")
        barcode_type = obj.type

        results.append({
            "type": barcode_type,
            "data": barcode_data,
            "rect": obj.rect
        })

    return results


# =========================
# 5. 전처리 후보 생성
# =========================

def preprocess_images(image):
    processed = []

    processed.append(("original", image))

    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    processed.append(("gray", gray))

    equalized = cv2.equalizeHist(gray)
    processed.append(("equalized", equalized))

    _, binary = cv2.threshold(gray, 120, 255, cv2.THRESH_BINARY)
    processed.append(("binary", binary))

    adaptive = cv2.adaptiveThreshold(
        gray,
        255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY,
        31,
        2
    )
    processed.append(("adaptive", adaptive))

    return processed


# =========================
# 6. 회전 + 전처리 통합 인식
# =========================

def read_barcode_robust(image):
    angles = [0, 90, 180, 270]

    for angle in angles:
        rotated = rotate_image(image, angle)
        candidates = preprocess_images(rotated)

        for preprocess_name, candidate in candidates:
            results = read_barcode(candidate)

            if results:
                return {
                    "success": True,
                    "angle": angle,
                    "preprocess": preprocess_name,
                    "results": results,
                    "rotated_image": rotated
                }

    return {
        "success": False,
        "angle": None,
        "preprocess": None,
        "results": [],
        "rotated_image": None
    }


# =========================
# 7. 인식 결과 표시
# =========================

def draw_barcode_result(image, results):
    output_image = image.copy()

    for r in results:
        rect = r["rect"]
        x, y, w, h = rect.left, rect.top, rect.width, rect.height

        cv2.rectangle(
            output_image,
            (x, y),
            (x + w, y + h),
            (0, 255, 0),
            3
        )

        cv2.putText(
            output_image,
            r["data"],
            (x, max(y - 10, 30)),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (0, 255, 0),
            2
        )

    return output_image


# =========================
# 8. 버튼 클릭 시 실행
# =========================

def on_run_button_clicked(b):
    with output:
        clear_output()

        try:
            image = load_uploaded_image(uploader)

            print("이미지 업로드 완료")
            print("바코드 인식 중...\n")

            result = read_barcode_robust(image)

            if result["success"]:
                print("바코드 인식 성공")
                print(f"회전 각도: {result['angle']}도")
                print(f"전처리 방식: {result['preprocess']}\n")

                for r in result["results"]:
                    print(f"바코드 종류: {r['type']}")
                    print(f"읽은 숫자: {r['data']}")

                output_image = draw_barcode_result(
                    result["rotated_image"],
                    result["results"]
                )

                plt.figure(figsize=(12, 8))
                plt.imshow(cv2.cvtColor(output_image, cv2.COLOR_BGR2RGB))
                plt.axis("off")
                plt.show()

            else:
                print("바코드 인식 실패")
                print("가능한 원인:")
                print("- 이미지 해상도가 낮음")
                print("- 바코드가 너무 작음")
                print("- 화면 캡처라 왜곡이 있음")
                print("- 바코드 주변 여백이 너무 큼")
                print("- 실제 바코드 영역만 크롭해야 할 수 있음")

        except Exception as e:
            print("오류 발생:")
            print(e)


run_button.on_click(on_run_button_clicked)

FileUpload(value=(), accept='.png,.jpg,.jpeg', description='Upload')

Button(button_style='success', description='바코드 읽기', style=ButtonStyle())

Output()